In [ ]:
!pip install -q google-genai

import json
import time
from google import genai
from google.colab import files

print("Reading local extracted_data.json...")
with open("extracted_data.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

client = genai.Client(api_key="NEVER GONNA GIVE YOU UP")
all_cooked_chunks = []

print("Resuming processing with safety delays...")

for idx, page_item in enumerate(raw_data):
    page_num = page_item["page"]
    page_content = page_item["content"]

    if len(page_content.strip()) < 10:
        continue

    prompt = f"""
You are a legal data engineer structuring a Thai traffic law document for a RAG system.
Clean and format this text from Page {page_num}.
Ensure main headers, sub-headers, and list items are grouped together logically in clean paragraphs. Do not lose any legal details or fines.
Output ONLY a valid JSON array of objects, where each object has:
- "source": "Page {page_num}"
- "type": "narrative" (or "fine_table" if it's a penalty list)
- "content": "the cleaned text string"

Raw text to process:
{page_content}
"""

    success = False
    retries = 3

    while retries > 0 and not success:
        try:
            response = client.models.generate_content(
                model='gemini-3.1-flash-lite',
                contents=prompt
            )
            clean_text = response.text.strip().removeprefix("```json").removesuffix("```").strip()
            page_chunks = json.loads(clean_text)

            all_cooked_chunks = [c for c in all_cooked_chunks if c["source"] != f"Page {page_num}"]

            if isinstance(page_chunks, list):
                all_cooked_chunks.extend(page_chunks)
            else:
                all_cooked_chunks.append({"source": f"Page {page_num}", "type": "narrative", "content": page_content})

            print(f"Successfully processed Page {page_num}")
            success = True
            time.sleep(2)

        except Exception as e:
            retries -= 1
            print(f"Retrying Page {page_num} due to server load... ({3 - retries}/3). Error: {e}")
            time.sleep(5)

all_cooked_chunks.sort(key=lambda x: int(x["source"].replace("Page ", "")))

output_filename = "rag_database.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(all_cooked_chunks, f, ensure_ascii=False, indent=2)

print(f"Cooking complete! Total clean chunks: {len(all_cooked_chunks)}")
files.download(output_filename)

Reading local extracted_data.json...
Cooking the data... this might take a minute.
Done! Downloading your new database.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>